In [824]:
import pandas as pd
import requests
import json

# rapidApi headers
headers = {
    "X-RapidAPI-Key": "REDACTED_RAPIDAPI_KEY",
    "X-RapidAPI-Host": "api-nba-v1.p.rapidapi.com"
}

#########################################################
# team code by id
url = "https://api-nba-v1.p.rapidapi.com/teams"
response = requests.get(url, headers=headers).json()['response']
df = pd.DataFrame(response)
df = df.loc[(df['nbaFranchise'] == True) & (df['allStar'] == False)]
nba_teams_df = df[['id', 'code']]
#########################################################

In [825]:
# display all nba teams with code
display(nba_teams_df)

,id,code
0,1,ATL
1,2,BOS
3,4,BKN
4,5,CHA
5,6,CHI
6,7,CLE
7,8,DAL
8,9,DEN
9,10,DET
10,11,GSW


In [826]:
#########################################################
# games by game ids #####################################
def get_games_by_game_ids(season, team):
    url = "https://api-nba-v1.p.rapidapi.com/games"
    querystring = {"season":season,"team":team}

    response = requests.get(url, headers=headers, params=querystring)
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    df = df.loc[(df["status.long"] == "Finished")]
    df = df.sort_values(by=["date.start"])
    
    # adding win col to df
    df['win'] = ''

    df.loc[
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = 1
    
    df.loc[
        ((df["scores.home.points"] < df["scores.visitors.points"]) & 
        (int(team) == df["teams.home.id"])) |
        ((df["scores.home.points"] > df["scores.visitors.points"]) & 
        (int(team) == df["teams.visitors.id"])),
        'win'
    ] = 0
    
    # adding home col to df
    df['home'] = ''
    
    df.loc[(int(team) == df["teams.home.id"]), 'home'] = 1
        
    df.loc[(int(team) == df["teams.visitors.id"]), 'home'] = 0    
    
    return drop_cols(df, cols_to_drop_for_game_stats)
#########################################################

In [827]:
cols_to_drop_for_player_stats = [
    'comment',
    'player.firstname',
    'player.lastname',
    'player.id',
    'team.id',
    'team.nickname',
    'team.code',
    'team.name',
    'team.logo',
    'game.id',
    'pos'
]

cols_to_drop_for_game_stats = [
    'league',
    'season',
    'stage',
    'officials',
    'timesTied',
    'leadChanges',
    'nugget',
    'date.start',
    'date.end',
    'date.duration',
    'status.clock',
    'status.halftime',
    'status.short',
    'status.long',
    'periods.current',
    'periods.total',
    'periods.endOfPeriod',
    'arena.name',
    'arena.city',
    'arena.state',
    'arena.country',
    'teams.visitors.id',
    'teams.visitors.name',
    'teams.visitors.nickname',
    'teams.visitors.code',
    'teams.visitors.logo',
    'teams.home.id',
    'teams.home.name',
    'teams.home.nickname',
    'teams.home.code',
    'teams.home.logo',
    'scores.visitors.win',
    'scores.visitors.loss',
    'scores.visitors.series.win',
    'scores.visitors.series.loss',
    'scores.visitors.linescore',
    'scores.home.win',
    'scores.home.loss',
    'scores.home.series.win',
    'scores.home.series.loss',
    'scores.home.linescore'
]

In [828]:
#########################################################
# drop all cols from a df ##############################
def drop_cols(df, cols):
    for col in cols:
        df = df.drop(col, axis=1)
    return df
#########################################################

In [829]:
#########################################################
# get player stats by game by
# num_mapping_table = str.maketrans({'-': '', '.': '', '+': ''})

def get_top_players_per_game_df(n, team, game_id):    
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"

    querystring = {"game": game_id }
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )

    df_opponent = df.loc[df['team.id'] != team]
    df = df.loc[df['team.id'] == team]
    
    df_opponent = drop_cols(df_opponent, cols_to_drop_for_player_stats)
    df = drop_cols(df, cols_to_drop_for_player_stats)
    
    df_opponent = zero_non_numeric_values(df_opponent)
    df = zero_non_numeric_values(df)

   
    df_opponent = df_opponent.add_prefix("opponent.")
    
#     df["plusMinus"] = df["plusMinus"].apply(lambda x: 
#         float(x) if x.translate(num_mapping_table).isnumeric() else 0.0)

    return { 
        "opponent": df_opponent.nlargest(n, "opponent.plusMinus"), 
        "friendly": df.nlargest(n, "plusMinus") 
    }
#########################################################

In [830]:
#########################################################
# flatten data frames ###################################
def flatten_df(df):
    # Flatten the DataFrame
    flattened_data = {}
    for col in df.columns:
        for row in range(df.shape[0]):
            new_col_name = f"{col}{row}"
            flattened_data[new_col_name] = df[col].iloc[row]

    # Convert to DataFrame
    return pd.DataFrame([flattened_data])
#########################################################

In [831]:
#########################################################
# per team, per game, get top 5 players by plusMinus metric
def get_top_five_players_per_game(team_id, game_ids): 
    top_5_players_on_team_per_game = {}
    for game_id in game_ids:
        # transform player_stats_df
        top_five_players_stats = get_top_players_per_game_df(5, team_id, game_id)
        opponent_top_five_player_stats = flatten_df(top_five_players_stats.get("opponent"))
        friendly_top_five_player_stats = flatten_df(top_five_players_stats.get("friendly"))
        top_five_players_stats = pd.concat(
            [opponent_top_five_player_stats, friendly_top_five_player_stats], 
            axis=1
        )
        ###########################
        top_5_players_on_team_per_game[game_id] = top_five_players_stats          
    return top_5_players_on_team_per_game
#########################################################

In [832]:
#########################################################
# get win prc, and last 10 win prc ######################
def get_win_prc(game_df):
    total_win_prc = []
    last_ten_win_prc = []
    last_ten_games_win_loss = []
    total_games_played = []
    last_ten_win_count = 0
    total_win_count = 0
    total_game_count = 0
    
    for index, row in game_df.iterrows():
        total_game_count += 1
        last_ten_games_win_loss.append(row['win'])
        
        if row['win']:
            total_win_count += 1
            last_ten_win_count += 1
            
        total_win_prc.append(total_win_count/total_game_count)
               
        if total_game_count >= 10:
            if last_ten_games_win_loss[0]:
                last_ten_win_count -= 1
            last_ten_games_win_loss.pop(0)
            last_ten_win_prc.append(last_ten_win_count/10)
        else:
            last_ten_win_prc.append(last_ten_win_count/total_game_count)
        
        total_games_played.append(total_game_count)            
    return last_ten_win_prc, total_win_prc, total_games_played
#########################################################

In [833]:
#########################################################
# combine player stats and games features ###############
import numpy as np
def combine_player_stats_and_games_data(games_df, player_stats_per_game):
    feature_map = []
    feature_map_cols_header = []
    for index, row in games_df.iterrows():
        game_id = row["id"]
        game_df = pd.DataFrame(row).transpose()
        players_stats_df = player_stats_per_game.get(game_id)
        if len(feature_map_cols_header) < 1:
            feature_map_cols_header = list(game_df) + list(players_stats_df)

        game_data = np.array(game_df.iloc[0])
        player_stats_data = np.array(players_stats_df.iloc[0])
        feature_map_data_row = np.concatenate((game_data, player_stats_data))
        feature_map.append(feature_map_data_row)
    return pd.DataFrame(feature_map, columns=feature_map_cols_header)
#########################################################

In [834]:
#########################################################
def get_feature_map_and_y(season, team_id):
    games_df = get_games_by_game_ids(season, team_id) 
    
    players_stats_per_games = get_top_five_players_per_game(
        team_id, 
        games_df["id"].array
    )
    
    last_ten_win_prc, total_win_prc, total_games_played = get_win_prc(games_df)
    games_df = games_df.assign(last_ten_w_prc=last_ten_win_prc)
    games_df = games_df.assign(w_prc=total_win_prc)
    games_df = games_df.assign(games_played=total_games_played)
    
    df = combine_player_stats_and_games_data(games_df, players_stats_per_games)
    df = zero_non_numeric_values(df)
    x = drop_cols(df, ["win", "id"])
    y = df["win"]
    return x, y 
#########################################################

In [835]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score

def get_rf_regressor_and_classifier_model(x, y):
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    # Regressor #############################################
    rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_regressor.fit(x_train, y_train)

    # Make predictions
    rf_regressor_predictions = rf_regressor.predict(x_test)
    
    # Evaluate the model
    rf_regressor_mse = mean_squared_error(y_test, rf_regressor_predictions)
    print(f'FR Regressor Mean Squared Error: {rf_regressor_mse}')
    #########################################################

    # Classifier ############################################
    # training random forest classifier 
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

    # Train the model on the training set
    rf_classifier.fit(x_train, y_train)
    
    # Make predictions on the test set
    rf_classifier_predictions = rf_classifier.predict(x_test)

    # Evaluate the model
    rf_classifier_accuracy = accuracy_score(y_test, rf_classifier_predictions)
    print(f'Accuracy: {rf_classifier_accuracy:.2f}')
    #########################################################
    
    return rf_regressor, rf_classifier


In [836]:
def zero_non_numeric_values(df):
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

In [880]:
def get_data_and_models(season, team_id): 
    x, y = get_feature_map_and_y(season, team_id)
    regressor, classifier = get_rf_regressor_and_classifier_model(x, y)
    return x, y, regressor, classifier

In [881]:
def get_player_season_stats_avgs(season, player_id):
    url = "https://api-nba-v1.p.rapidapi.com/players/statistics"
    querystring = {"id":player_id,"season":season}
    
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    df = drop_cols(df, cols_to_drop_for_player_stats)
    df = zero_non_numeric_values(df) 
    
    return df.mean()
    
def get_players_season_stats_avgs(season, player_ids):
    player_stats_avgs = []
    for i in player_ids:
        player_stats_avgs.append(get_player_season_stats_avgs(season, i))
    
    df = pd.DataFrame(player_stats_avgs)
    return df.sort_values(by=["plusMinus"], ascending=False)


In [882]:
# scores.visitors.points,scores.home.points,home,last_ten_w_prc,w_prc,games_played
def get_prediction_feature_map(
    training_data_tail,
    season,
    team_id,
    o_team_id,
    home_avg_ppg,
    visitors_avg_ppg,
    home,
    name_1,
    name_2,
    name_3,
    name_4,
    name_5,
    o_name_1,
    o_name_2,
    o_name_3,
    o_name_4,
    o_name_5
):
    
    name_set = { 
        name_1,
        name_2,
        name_3,
        name_4,
        name_5
    }
    
    o_name_set = {
        o_name_1,
        o_name_2,
        o_name_3,
        o_name_4,
        o_name_5
    }
     
    url = "https://api-nba-v1.p.rapidapi.com/players"
    querystring = {"team":team_id,"season":season}
    df = pd.json_normalize(
        requests.get(url, headers=headers, params=querystring)
        .json()["response"]
    )
    
    o_querystring = {"team":o_team_id,"season":season}
    o_df = pd.json_normalize(
        requests.get(url, headers=headers, params=o_querystring)
        .json()["response"]
    )

    concat_names = df['firstname'] + " " + df['lastname']
    o_concat_names = o_df['firstname'] + " " + o_df['lastname']
    
    # Filter the DataFrame based on whether the concatenated names exist in name sets
    df = df[concat_names.isin(name_set)] 
    o_df = o_df[o_concat_names.isin(o_name_set)]
    
    player_ids = np.array(df["id"])
    o_player_ids = np.array(o_df["id"])
    
    players_stats_avgs = get_players_season_stats_avgs(season, player_ids)
    o_players_stats_avgs = get_players_season_stats_avgs(season, o_player_ids)
    o_players_stats_avgs = opponent_players_stats_avgs.add_prefix("opponent.")

    o_players_stats_avgs = flatten_df(opponent)
    players_stats_avgs = flatten_df(friendly)

    players_stats = pd.concat(
        [opponent_players_stats_avgs, friendly_players_stats_avgs], 
        axis=1
    )
    
    games_col_headers = [
        "scores.visitors.points",
        "scores.home.points", 
        "home", 
        "last_ten_w_prc", 
        "w_prc", 
        "games_played"
    ]
    
    games_data = [
        visitors_avg_ppg, 
        home_avg_ppg, 
        home, 
        training_data_tail["last_ten_w_prc"].iloc[0], 
        training_data_tail["w_prc"].iloc[0],
        training_data_tail["games_played"].iloc[0] + 1
    ]
        
    feature_map_col_headers = games_col_headers + list(players_stats)
    feature_map_data_row = np.concatenate((games_data, players_stats.iloc[0]))
    return pd.DataFrame([feature_map_data_row], columns=feature_map_col_headers)    

In [840]:
print("Hornets")
hornets_x, hornets_y, hornets_regressor, hornests_classifier = get_data_and_models("2023", 5)

print("Pistons")
pistons_x, pistons_y, pistons_regressor, pistons_classifier = get_data_and_models("2023", 10)

print("Suns")
suns_x, sun_y, suns_regressor, suns_classifier = get_data_and_models("2023", 28)

print("Cavs")
cavs_x, cavs_y, cavs_regressor, cavs_classifier = get_data_and_models("2023", 7)

print("Warrios")
war_x, war_y, war_regressor, war_classifier = get_data_and_models("2023", 11)

print("Spurs")
spurs_x, spurs_y, spurs_regressor, spurs_classifier = get_data_and_models("2023", 31)

print("Mavs")
mavs_x, mavs_y, mavs_regressor, mavs_classifier = get_data_and_models("2023", 8)

print("Bulls")
bulls_x, bulls_y, bulls_regressor, bulls_classifier = get_data_and_models("2023", 6)

print("Raptors")
rapt_x, rapt_y, rapt_regressor, rapt_classifier = get_data_and_models("2023", 38)

print("Nuggets")
nugs_x, nugs_y, nugs_regressor, nugs_classifier = get_data_and_models("2023", 9)

print("Celtics")
bos_x, bos_y, bos_regressor, bos_classifier = get_data_and_models("2023", 2)

print("Blazers")
blazers_x, blazers_y, blazers_regressor, blazers_classifier = get_data_and_models("2023", 29)

Hornets
FR Regressor Mean Squared Error: 0.006100000000000002
Accuracy: 1.00
Pistons
FR Regressor Mean Squared Error: 0.06559999999999999
Accuracy: 1.00
Suns
FR Regressor Mean Squared Error: 0.08424285714285715
Accuracy: 0.86
Cavs
FR Regressor Mean Squared Error: 0.07908571428571429
Accuracy: 0.93
Warrios
FR Regressor Mean Squared Error: 0.055764285714285725
Accuracy: 0.93
Spurs
FR Regressor Mean Squared Error: 0.11755714285714285
Accuracy: 0.93
Mavs
FR Regressor Mean Squared Error: 0.050364285714285716
Accuracy: 0.93
Bulls
FR Regressor Mean Squared Error: 0.1112642857142857
Accuracy: 0.86
Raptors
FR Regressor Mean Squared Error: 0.08294285714285714
Accuracy: 0.93
Nuggets
FR Regressor Mean Squared Error: 0.05255
Accuracy: 0.86
Celtics
FR Regressor Mean Squared Error: 0.07147142857142856
Accuracy: 0.86
Blazers
FR Regressor Mean Squared Error: 0.06184285714285715
Accuracy: 0.93


In [887]:
hornets_pred = get_prediction_feature_map(
    hornets_x.tail(1),
    "2023",
    5,
    10,
    112.3,
    107.2,
    0,
    "Vasilije Micic",
    "Grant Williams",
    "Brandon Miller",
    "Miles Bridges",
    "Nick Richards",
    "Cade Cunningham",
    "Jaden Ivey",
    "Evan Fournier",
    "Isaiah Stewart",
    "Jalen Duren"
)

pistons_pred = get_prediction_feature_map(
    pistons_x.tail(1),
    "2023",
    10,
    5,
    112.3,
    107.2,
    1,
    "Cade Cunningham",
    "Jaden Ivey",
    "Evan Fournier",
    "Isaiah Stewart",
    "Jalen Duren",
    "Vasilije Micic",
    "Grant Williams",
    "Brandon Miller",
    "Miles Bridges",
    "Nick Richards"
)

In [888]:
print("hornets regressor : ", hornets_regressor.predict(hornets_pred))
print("pistons regressor : ", pistons_regressor.predict(pistons_pred))
print("hornets classifier : ", hornests_classifier.predict(hornets_pred))
print("pistons classifier : ", pistons_classifier.predict(pistons_pred))

hornets regressor :  [0.02]
pistons regressor :  [0.22]
hornets classifier :  [0]
pistons classifier :  [0]


In [889]:
suns_pred = get_prediction_feature_map(
    suns_x.tail(1),
    "2023",
    28,
    7,
    113.8,
    117,
    0,
    "Devin Booker",
    "Bradley Beal",
    "Grayson Allen",
    "Kevin Durant",
    "Jusuf Nurkic",
    "Darius Garland",
    "Caris LeVert",
    "Isaac Okoro",
    "George Niang",
    "Jarrett Allen"
)

cavs_pred = get_prediction_feature_map(
    cavs_x.tail(1),
    "2023",
    7,
    28,
    113.8,
    117,
    1,
    "Darius Garland",
    "Caris LeVert",
    "Isaac Okoro",
    "George Niang",
    "Jarrett Allen",
    "Devin Booker",
    "Bradley Beal",
    "Grayson Allen",
    "Kevin Durant",
    "Jusuf Nurkic",
)

In [890]:
print("suns regressor : ", suns_regressor.predict(suns_pred))
print("cavs regressor : ", cavs_regressor.predict(cavs_pred))
print("suns classifier : ", suns_classifier.predict(suns_pred))
print("cavs classifier : ", cavs_classifier.predict(cavs_pred))

suns regressor :  [0.18]
cavs regressor :  [0.12]
suns classifier :  [0]
cavs classifier :  [0]


In [891]:
war_pred = get_prediction_feature_map(
    war_x.tail(1),
    "2023",
    11,
    31,
    112.6,
    118.7,
    0,
    "Chris Paul",
    "Brandin Podziemski",
    "Andrew Wiggins",
    "Jonathan Kuminga",
    "Draymond Green",
    "Tre Jones",
    "Devin Vassell",
    "Julian Champagnie",
    "Jeremy Sochan",
    "Victor Wembanyama",  
)

spurs_pred = get_prediction_feature_map(
    spurs_x.tail(1),
    "2023",
    31,
    11,
    112.6,
    118.7,
    1,
    "Tre Jones",
    "Devin Vassell",
    "Julian Champagnie",
    "Jeremy Sochan",
    "Victor Wembanyama",
    "Chris Paul",
    "Brandin Podziemski",
    "Andrew Wiggins",
    "Jonathan Kuminga",
    "Draymond Green"
)

In [892]:
print("spurs classifier : ", spurs_classifier.predict(spurs_pred))
print("war classifier : ", war_classifier.predict(war_pred))
print("spurs regressor : ", spurs_regressor.predict(spurs_pred))
print("war regressor : ", war_regressor.predict(war_pred))

spurs classifier :  [0]
war classifier :  [1]
spurs regressor :  [0.87]
war regressor :  [0.68]


In [893]:
print("7")
mavs_pred = get_prediction_feature_map(
    mavs_x.tail(1),
    "2023",
    8,
    6,
    111.8,
    119,
    0,
    "Luka Doncic",
    "Kyrie Irving",
    "Derrick Jones Jr.",
    "P.J. Washington",
    "Daniel Gafford",
    "Coby White",
    "Ayo Dosunmu",
    "Alex Caruso",
    "DeMar DeRozan",
    "Nikola Vucevic"
)

print("8")
bulls_pred = get_prediction_feature_map(
    bulls_x.tail(1),
    "2023",
    6,
    8,
    111.8,
    119,
    1,
    "Coby White",
    "Ayo Dosunmu",
    "Alex Caruso",
    "DeMar DeRozan",
    "Nikola Vucevic",
    "Luka Doncic",
    "Kyrie Irving",
    "Derrick Jones Jr.",
    "P.J. Washington",
    "Daniel Gafford"
)

7
8


In [894]:
print("mavs classifier : ", mavs_classifier.predict(mavs_pred))
print("bulls classifier : ", bulls_classifier.predict(bulls_pred))
print("mavs regressor : ", mavs_regressor.predict(mavs_pred))
print("bulls regressor : ", bulls_regressor.predict(bulls_pred))

mavs classifier :  [1]
bulls classifier :  [0]
mavs regressor :  [0.45]
bulls regressor :  [0.41]


In [895]:
rapt_pred = get_prediction_feature_map(
    rapt_x.tail(1),
    "2023",
    38,
    9,
    114.9,
    114.2,
    0,
    "Bruce Brown",
    "RJ Barrett",
    "Kelly Olynyk",
    "Gradey Dick",
    "Ochai Agbaji",
    "Jamal Murray",
    "Kentavious Caldwell-Pope",
    "Michael Porter Jr.",
    "Aaron Gordon",
    "Nikola Jokic"
)

nugs_pred = get_prediction_feature_map(
    nugs_x.tail(1),
    "2023",
    9,
    38,
    114.9,
    114.2,
    1,
    "Jamal Murray",
    "Kentavious Caldwell-Pope",
    "Michael Porter Jr.",
    "Aaron Gordon",
    "Nikola Jokic",
    "Bruce Brown",
    "RJ Barrett",
    "Kelly Olynyk",
    "Gradey Dick",
    "Ochai Agbaji"
)

In [896]:
print("rapt classifier: ", rapt_classifier.predict(rapt_pred))
print("nugs classifier: ", nugs_classifier.predict(nugs_pred))
print("rapt regressor: ", rapt_regressor.predict(rapt_pred))
print("nugs regressor: ", nugs_regressor.predict(nugs_pred))

rapt classifier:  [0]
nugs classifier:  [1]
rapt regressor:  [0.25]
nugs regressor:  [0.27]


In [899]:
blazers_pred = get_prediction_feature_map(
    blazers_x.tail(1),
    "2023",
    29,
    2,
    108.1,
    120.7,
    1,
    "Scoot Henderson",
    "Anfernee Simons",
    "Jerami Grant",
    "Toumani Camara",
    "Deandre Ayton",
    "Payton Pritchard",
    "Derrick White",
    "Jaylen Brown",
    "Jayson Tatum",
    "Kristaps Porzingis"
)

bos_pred = get_prediction_feature_map(
    bos_x.tail(1),
    "2023",
    2,
    29,
    108.1,
    120.7,
    0,
    "Payton Pritchard",
    "Derrick White",
    "Jaylen Brown",
    "Jayson Tatum",
    "Kristaps Porzingis",
    "Scoot Henderson",
    "Anfernee Simons",
    "Jerami Grant",
    "Toumani Camara",
    "Deandre Ayton"
)

In [900]:
print("blazers classifier: ", blazers_classifier.predict(blazers_pred))
print("bos classifier: ", bos_classifier.predict(bos_pred))
print("blazers regressor: ", blazers_regressor.predict(blazers_pred))
print("bos regressor: ", bos_regressor.predict(bos_pred))

blazers classifier:  [0]
bos classifier:  [1]
blazers regressor:  [0.47]
bos regressor:  [0.58]
